# Laboratorio 9. Persistencia ligera con sqlite3

Creación de una base SQLite local, inserción de registros y consulta parametrizada.

## 1. Preparar rutas del tema

In [ ]:
from pathlib import Path


def localizar_tema10() -> Path:
    """Localiza el directorio Tema10 aunque el notebook se ejecute desde notebooks, Tema10 o Curso_Python."""
    actual = Path.cwd().resolve()
    candidatos = [
        actual,
        actual.parent,
        actual / "Tema10",
        actual.parent / "Tema10",
    ]

    for candidato in candidatos:
        if candidato.name == "Tema10" and candidato.exists():
            return candidato
        if (candidato / "src").exists() and (candidato / "notebooks").exists():
            return candidato

    raise RuntimeError("No se ha podido localizar el directorio Tema10. Abre el notebook desde Tema10/notebooks o desde el proyecto Curso_Python.")


base_dir = localizar_tema10()
src_dir = base_dir / "src"
data_dir = base_dir / "data"
output_dir = base_dir / "output"
backup_dir = base_dir / "backup"
log_dir = base_dir / "logs"

for directorio in (data_dir, output_dir, backup_dir, log_dir):
    directorio.mkdir(parents=True, exist_ok=True)

print("base_dir:", base_dir)
print("data_dir:", data_dir)

## 2. Importar módulos y abrir base de datos

In [ ]:
import sqlite3
from pathlib import Path

db_file = data_dir / "inventario.db"
bd = sqlite3.connect(db_file)
print("Base SQLite:", db_file)

## 3. Crear tabla si no existe

In [ ]:
bd.execute("""
CREATE TABLE IF NOT EXISTS nodos (
    host TEXT,
    ip TEXT,
    rol TEXT
)
""")

## 4. Insertar datos de forma reproducible

In [ ]:
bd.execute("DELETE FROM nodos")

datos = [
    ("web-01", "10.0.1.10", "frontend"),
    ("db-01", "10.0.1.20", "backend"),
    ("cache-01", "10.0.1.30", "redis"),
]

bd.executemany("INSERT INTO nodos VALUES (?, ?, ?)", datos)
bd.commit()
print("Registros insertados:", len(datos))

## 5. Consultar con parámetros

Los valores externos deben pasarse con parámetros, no concatenando SQL.

In [ ]:
for fila in bd.execute("SELECT host, ip FROM nodos WHERE rol = ?", ("backend",)):
    print(f"Servidor crítico: {fila[0]} localizado en {fila[1]}")

## 6. Cerrar la conexión

In [ ]:
bd.close()
print("Conexión cerrada")